# Generate commets for Sustainability Posts and similairty checks

In [1]:

import pandas as pd
import re
import nltk
import warnings

warnings.filterwarnings("ignore")

# Load the file and make similar prompts
df= pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\WP2_Simulation_Tool\data\final_data_cleaned_with_author_names.csv")

df["count of sentences"] = df["text"].apply(lambda x: len(nltk.sent_tokenize(str(x))))
dataframe = df.groupby(["text_ha"]).agg({"text_ha": 'first',
                                          'text': lambda x: len(x),
                                          'count of sentences': lambda x: list(x)})


In [2]:
dataframe

,text_ha,text,count of sentences
text_ha,,,
"""Als ik de buurt en het klimaat hiermee kan helpen, waarom niet?"" Dat antwoordde Dylan toen hem werd gevraagd of hij mee wilde doen met de renovatie van het flatgebouw waar hij woont. ""Het scheelt me straks ook ongeveer 15 euro op de energierekening."" De 24-jarige Dylan vertelt over de renovatie. Tot nu toe is hij tevreden, want het begon al goed. ""De aannemer kwam persoonlijk langs om de planning uit te leggen. Dat was fijn, want daardoor was alles meteen heel duidelijk."" Deze aannemer van Exterio/Lenferink vertelde dat bij alle woningen de muren worden geisoleerd, dat er nieuw glas in de ramen komt en dat er nieuwe voor- en achterdeuren komen. ""Ze beginnen met de ramen in de slaapkamer, keuken en woonkamer,"" vertelt Dylan. ""Als dat klaar is komt er een nieuwe voordeur en balkondeur."" De deuren vindt Dylan het mooist. ""Die mochten we ook zelf kiezen. Ik kreeg een brief met vier type voordeuren. Eerst zouden de deuren roodbruin blijven, maar na suggesties van de bewoners is besloten dat de deuren wit worden."" Hoe zal de verbouwing verder gaan? Blijft Dylan enthousiast? De komende tijd lees je er alles over! - Lekker Bomen gaat over mensen die helpen om de wereld een stukje groener te maken. Deze maand staat Dylan centraal! We werken samen met Reimarkt aan de verduurzaming van onze woningen: https://bit.ly/3dKE7m1","""Als ik de buurt en het klimaat hiermee kan he...",3,"[1, 3, 2]"
"""Een huis huur je, maar een thuis maak je zelf"", aldus de gelukkige Jose over haar CO2-neutrale huurwoning. Sinds kort zijn haar dochter en kleinzoon het huis uit en was het tijd om haar huis helemaal eigen te maken. Ook bij Accolade verduurzamen we volop. Zo hebben onze huurhuizen eind dit jaar gemiddeld energielabel B. Benieuwd naar het verhaal van Jose? https://bit.ly/2DizVKl #WonenDoenWeSamen #woningcorporaties","""Een huis huur je, maar een thuis maak je zelf...",6,"[1, 4, 1, 8, 1, 6]"
"""Had het niet verwacht, maar ik heb toch nog iets kunnen besparen."" #energieadviseur komt langs Huurdersbelangenvereniging Etten-Leur https://www.alwel.nl/alwel-belicht/Nieuws/3224/niet-verwacht-maar-heb-toch-iets-kunnen-besparen/","""Had het niet verwacht, maar ik heb toch nog i...",1,[3]
"""Hallo, mijn naam is Annelotte. Voor mijn Bachelor Eindproject aan de TU Delft doe ik onderzoek naar de verduurzaming van sociale huurwoningen. Woningcorporaties zijn hier druk mee bezig en het is belangrijk dat huurders dit ook goed vinden! Daarom heb ik een enquete gemaakt om erachter te komen wat huurders tegenhoudt bij het instemmen met plannen om hun woning te verduurzamen. Laat mij weten wat u hiervan vindt door deze enquete voor mij in te vullen. Het kost u maar 4 minuten en het helpt mij heel erg met mijn onderzoek! Alvast bedankt!"" Link naar de enquete: https://tudelft.fra1.qualtrics.com/jfe/form/SV_8J18hx1rCjOKiGx","""Hallo, mijn naam is Annelotte. Voor mijn Bach...",2,"[1, 2]"
"""Ik ben heel blij met mijn appartement en mooi dat het nog energieneutraal is ook."" www.bit.ly/huuskestek Jasper neemt de sleutels van zijn nieuwe appartement in het Huuske van Hooghkamer in Voorhout in ontvangst. Het gebouw is optimaal geisoleerd en wekt zelf energie op.","""Ik ben heel blij met mijn appartement en mooi...",1,[16]
...,...,...,...
"de Schouw 17 RAALTE RAALTE: De bezichtiging van deze woning vindt plaats op dinsdag om 21 april om 09:00 uur met een medewerkster van SallandWonen en de uitgenodigde kandidaat. De datum van sleutel overdracht is ondervoorbehoud. De bruto huur is inclusief EUR 42,60 voorschotbedrag voor zonnestroom.",de Schouw 17 RAALTE RAALTE: De bezichtiging va...,1,[1]
"de Schouw 48 RAALTE RAALTE: De bezichtiging van deze woning vindt plaats in overleg met de huidige huurder en de uitgenodigde kandidaat. Deze woning verhuren we met 6 zonnepanelen. Het voorschotbedrag voor zonnestroom bedraagt EUR 24,61 per maand.",de Schouw 48 RAALTE RAALTE: De bezichtiging va...,5,"[1, 1, 1, 1, 1]"
"de Spinde 49 RA

In [ ]:
import re
import torch
import pandas as pd

from transformers import (
    Mistral3ForConditionalGeneration,
    MistralCommonBackend,
)

# -------------------------------------------------------------------
# 1. Model & tokenizer setup
# -------------------------------------------------------------------

model_id = "mistralai/Ministral-3-3B-Instruct-2512"

# 👉 Use your real HF token string here or rely on env var
import os
hf_token = os.environ.get("HF_TOKEN", "")  # <--- set HF_TOKEN env var, or replace with your token locally (do not commit tokens)

# Load tokenizer via MistralCommonBackend
tokenizer = MistralCommonBackend.from_pretrained(
    model_id,
    token=hf_token,
)

# Load model (FP8-backed, no need for load_in_8bit or trust_remote_code)
model = Mistral3ForConditionalGeneration.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",
)
model.eval()

# -------------------------------------------------------------------
# 2. Simple chatbot wrapper to mimic pipeline("text-generation", ...)
# -------------------------------------------------------------------

def chatbot(prompt: str, max_new_tokens: int = 80, top_k: int = 50, top_p: float = 0.85):
    """
    Emulates a text-generation pipeline for the Mistral 3 instruct model.
    Returns a list with one dict: [{"generated_text": <string>}],
    so the rest of your code can stay the same.
    """

    # Chat-style input: single user message with your full prompt
    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    # Prepare inputs using the chat template
    inputs = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
        add_generation_prompt=True,
    )

    # Move tensors to model device (GPU/CPU)
    inputs = {
        k: v.to(model.device) if isinstance(v, torch.Tensor) else v
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            top_p=top_p,
        )

    # Take only the generated part (strip the prompt tokens)
    generated_tokens = outputs[0, inputs["input_ids"].shape[-1]:]
    text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    # Match the old pipeline API
    return [{"generated_text": text}]


# -------------------------------------------------------------------
# 3. Your data + generation loop (unchanged logic)
# -------------------------------------------------------------------

# Example: dataframe must already exist and have these columns:
# dataframe = pd.read_csv("...")  # for example
# Expected columns: "text_ha", "text" (number of comments), "count of sentences"

new_data_comments_generated = pd.DataFrame(columns=["Post", "Comments"])

total_rows = 0
total_matches = 0
max_attempts = 3  # Maximum number of attempts to generate a comment

for i in range(len(dataframe)):

    print(
        "I:", i,
        "POST:", dataframe["text_ha"][i],
        "Number of comments:", dataframe["text"][i],
        "Count of sentences", dataframe["count of sentences"][i],
    )

    # Generate N comments per post
    for j in range(dataframe["text"][i]):

        example_post = (
            "'Schoonmaken. Opruimen. Foxhol winterklaar maken. Doe je mee?' Dat was het motto "
            "om de huurders in Foxhol aan te moedigen om in actie te komen. In Midden-Groningen "
            "houden we jaarlijks een actiedag waarin de leefbaarheid, in een wijk of dorp waar "
            "we veel woningen bezitten, centraal staat. Dit keer was Foxhol aan de beurt. Drie "
            "containers met een inhoud van veertig kuub lagen aan het eind van de middag vol "
            "met grof vuil! Tijdens de opruiming werden de broodjes hamburger liefdevol in "
            "ontvangst genomen door de bewoners, gepaard met koffie en thee. Al met al een "
            "succesvolle dag!"
        )

        example_comments = [
            "Mag men ook wel eens in Musselkanaal doen....top actie…",
            "Zorg eerst maar Dat .. West indischekade .. opgeruimd en netjes word .. dat zijn in totaal 3 straten 3 flatten Suc6 ..  ik steek me poten niet uit als jullie niet eens op of om kijken",
            "Best wel jammer, oud jaar komt er weer aan. Je had er lekker warm bij kunnen zitten met al dat hout en matrassen",
        ]

        prompts = f"""
Je bent een Nederlandstalige Facebookgebruiker die reageert op berichten van een woningcorporatie.

TAKEN:
1. Lees het onderstaande Facebookbericht.
2. Genereer ÉÉN natuurlijke Facebookreactie in het Nederlands.
3. De stijl en toon mogen lijken op de voorbeeldreacties (informele, echte reacties van bewoners).
4. Gebruik 1 tot 3 zinnen.

BELANGRIJK:
- Antwoord uitsluitend met de reactie zelf.
- Zet de volledige reactie tussen dubbele aanhalingstekens.
- Voeg GEEN extra tekst toe (geen uitleg, geen labels, geen opsomming, geen extra regels).
- Geen aanhef (zoals "Beste ...") en geen afsluiting (zoals "Groetjes").

Hier is het bericht waarop je reageert:
---
{dataframe["text_ha"][i]}
---

Hier zijn voorbeelden van een bericht en reacties:

Voorbeeldbericht:
"{example_post}"

Voorbeeldreactie 1:
"{example_comments[0]}"

Voorbeeldreactie 2:
"{example_comments[1]}"

Voorbeeldreactie 3:
"{example_comments[2]}"

Nu jij. Geef slechts ÉÉN nieuwe reactie, alleen de reactie, tussen dubbele aanhalingstekens:
"""

        attempts = 0
        match_found = False

        while not match_found and attempts < max_attempts:
            try:
                # Generate response using our custom chatbot
                response = chatbot(prompts, max_new_tokens=80, top_k=50, top_p=0.85)
                generated_text = response[0]["generated_text"].strip()
                print("Generated Text:", generated_text)

                # Extract text inside the last pair of double quotes
                pattern = r'"([^"]+)"'
                matches = re.findall(pattern, generated_text)

                if matches:
                    comment = matches[-1].strip()  # take last match (most reliable)
                    print("Match found:", comment)

                    new_row = [dataframe["text_ha"][i], comment]
                    new_data_comments_generated.loc[len(new_data_comments_generated)] = new_row

                    total_matches += 1
                    match_found = True
                else:
                    print(f"No match found in attempt {attempts + 1}")
                    attempts += 1

            except Exception as e:
                print(f"Error generating comment for post index {i}, attempt {attempts + 1}: {e}")
                attempts += 1

        # If no match is found after max_attempts, log it as 'No match found'
        if not match_found:
            print(f"Max attempts reached for post index {i}. Logging as 'No match found'.")
            new_row = [dataframe["text_ha"][i], "No match found"]
            new_data_comments_generated.loc[len(new_data_comments_generated)] = new_row

    print("Total matches:", total_matches)


In [ ]:
output_path = r"C:\Users\20245179\OneDrive - TU Eindhoven\LLM_EngD_project\Few Shot Prompt\Mistral_3_3B_instruct\Generated_comments_Mistral_3_3B_Instruct.csv"
new_data_comments_generated.to_csv(output_path, index=False)

In [ ]:
import pandas as pd
comments_df = pd.read_csv(r"C:\Users\20245179\OneDrive - TU Eindhoven\LLM_EngD_project\Generation and Evaluation\Sustainability_Posts_generated_Comments\Sustainability_Posts_Generated_comments_LLAMA_3_8B_Instruct_FEW_SHOT__1.csv")

In [ ]:
# remove comments with "No match found"
comments_df = comments_df[comments_df["Comments"] != "No match found"]

In [ ]:
comments_df

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

grouped = comments_df.groupby("Post")["Comments"].apply(list).reset_index(name="Comments")

# Encode posts and all comments
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
post_embeddings = model.encode(grouped["Post"].tolist(), convert_to_tensor=True, normalize_embeddings=True)

# Flatten comment list for encoding
all_comments = [comment for group in grouped["Comments"] for comment in group]
comment_embeddings = model.encode(all_comments, convert_to_tensor=True, normalize_embeddings=True)

# Compute real similarities (average per post)
real_avg_similarities = []

comment_cursor = 0

for i, comments in enumerate(grouped["Comments"]):
    n_comments = len(comments)
    post_embed = post_embeddings[i].unsqueeze(0)
    comment_embeds = comment_embeddings[comment_cursor: comment_cursor + n_comments]
    comment_cursor += n_comments

    sims = util.cos_sim(post_embed, comment_embeds).squeeze().cpu().numpy()
    real_avg_similarities.append(sims.mean())

real_avg_similarities = np.array(real_avg_similarities)

print(f"Real Avg Similarity: {real_avg_similarities.mean():.4f}")

# Random Sampling with Comment Size Consideration
num_simulations = 1000
random_avg_similarities = []
all_random_similarities = []
for _ in tqdm(range(num_simulations)):
    comment_cursor = 0
    sim_list = []

    for i, comments in enumerate(grouped["Comments"]):
        n_comments = len(comments)
        post_embed = post_embeddings[i].unsqueeze(0)

        # Randomly sample comments while maintaining comment size
        random_indices = np.random.choice(len(all_comments), size=n_comments, replace=False)
        sampled_comments = comment_embeddings[random_indices]

        sims = util.cos_sim(post_embed, sampled_comments).squeeze().cpu().numpy()

        # Ensure sims is always 1D
        sims = np.atleast_1d(sims)

        # Collect individual similarities
        all_random_similarities.extend(sims)
        sim_list.append(sims.mean())

    random_avg_similarities.append(np.mean(sim_list))

null_distribution = np.array(random_avg_similarities)

# Calculate p-value
p_value = np.mean(null_distribution >= real_avg_similarities.mean())
# Convert to numpy array
all_random_similarities = np.array(all_random_similarities)

# Plot the distribution
plt.figure(figsize=(14, 7))
sns.kdeplot(all_random_similarities, fill=True, color='skyblue', alpha=0.6, label='Individual Random Similarities')
plt.axvline(all_random_similarities.mean(), color='blue', linestyle='--', label=f'Random Mean: {all_random_similarities.mean():.4f}')

plt.title('Distribution of Individual Random Similarities')
plt.xlabel('Cosine Similarity')
plt.ylabel('Density')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Random Avg Similarity: {null_distribution.mean():.4f}")
print(f"Empirical p-value: {p_value:.4f}")


import matplotlib.pyplot as plt
import seaborn as sns

# Calculate means
real_mean = real_avg_similarities.mean()
random_mean = all_random_similarities.mean()

# Plot setup
plt.figure(figsize=(14, 7))

# KDE for All Random Similarities
sns.kdeplot(all_random_similarities, fill=True, color='skyblue', alpha=0.6, label='All Random Similarities')
plt.axvline(random_mean, color='blue', linestyle='--', label=f'Random Mean: {random_mean:.4f}')

# KDE for Real Similarities
sns.kdeplot(real_avg_similarities, fill=True, color='orange', alpha=0.6, label='Real Similarities')
plt.axvline(real_mean, color='orange', linestyle='--', label=f'Real Mean: {real_mean:.4f}')

# Plot settings
plt.title('Distribution of Real vs. All Random Similarities')
plt.xlabel('Cosine Similarity')
plt.ylabel('Density (Normalized)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()